## Cell 1: Check GPU Availability

In [ ]:
import json
import subprocess
import sys
import textwrap
from shutil import which


def _run(cmd: list[str], timeout_s: int = 10) -> tuple[int, str, str]:
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)
    return int(p.returncode), (p.stdout or ""), (p.stderr or "")


def _probe_nvidia_smi(timeout_s: int = 5) -> dict:
    """Fast GPU presence probe that doesn't require torch."""
    if which("nvidia-smi") is None:
        return {"nvidia_smi": False}
    try:
        rc, out, err = _run(["nvidia-smi", "-L"], timeout_s=timeout_s)
        txt = (out or err).strip()
        return {"nvidia_smi": True, "rc": rc, "raw": txt}
    except subprocess.TimeoutExpired:
        return {"nvidia_smi": True, "rc": -1, "raw": "nvidia-smi timed out"}
    except Exception as e:
        return {"nvidia_smi": True, "rc": -2, "raw": str(e)}


def _probe_torch_cuda(timeout_s: int = 20) -> dict:
    """Probe torch + CUDA in a separate process so a wedged CUDA init can't hang the notebook."""
    code = textwrap.dedent("""
    import json
    out = {}
    try:
        import torch
        out["torch_available"] = True
        out["torch_version"] = getattr(torch, "__version__", None)
        out["cuda_available"] = bool(torch.cuda.is_available())
        out["cuda_version"] = getattr(getattr(torch, "version", None), "cuda", None)
        if out["cuda_available"]:
            try:
                out["gpu"] = torch.cuda.get_device_name(0)
                out["gpu_mem_gb"] = torch.cuda.get_device_properties(0).total_memory / 1e9
            except Exception as e:
                out["gpu_error"] = str(e)
    except Exception as e:
        out["torch_available"] = False
        out["error"] = str(e)
    print(json.dumps(out))
    """)
    try:
        p = subprocess.run(
        [sys.executable, "-c", code],
        capture_output=True,
        text=True,
        timeout=timeout_s,
        )
    except subprocess.TimeoutExpired:
        return {"torch_available": False, "cuda_available": False, "error": f"torch/CUDA probe timed out after {timeout_s}s"}
    if p.returncode != 0:
        return {"torch_available": False, "cuda_available": False, "error": (p.stderr or p.stdout or "unknown error").strip()}
    try:
        return json.loads((p.stdout or "{}").strip() or "{}")
    except Exception as e:
        return {"torch_available": False, "cuda_available": False, "error": f"Failed to parse probe output: {e}", "raw": (p.stdout or "").strip()}


smi = _probe_nvidia_smi(timeout_s=5)
probe = _probe_torch_cuda(timeout_s=20)

TORCH_AVAILABLE = bool(probe.get("torch_available", False))
TORCH_VERSION = probe.get("torch_version") if TORCH_AVAILABLE else None
CUDA_AVAILABLE = bool(probe.get("cuda_available", False))

print("GPU/Runtime probe:")
print(f"  nvidia-smi present: {bool(smi.get('nvidia_smi', False))}")
if smi.get("nvidia_smi"):
    print(f"  nvidia-smi -L: {smi.get('raw')}")
print(f"  torch available: {TORCH_AVAILABLE}")
if TORCH_AVAILABLE:
    print(f"  torch version: {TORCH_VERSION}")
    print(f"  torch CUDA available: {CUDA_AVAILABLE}")
    if probe.get("cuda_version") is not None:
        print(f"  torch CUDA version: {probe.get('cuda_version')}")
    if CUDA_AVAILABLE:
        if probe.get("gpu"):
            print(f"  GPU: {probe.get('gpu')}")
        if probe.get("gpu_mem_gb") is not None:
            print(f"  GPU Memory: {float(probe['gpu_mem_gb']):.2f} GB")
else:
    if probe.get("error"):
        print(f"  torch import/probe note: {probe['error']}")
    print("  Note: torch is not installed yet in this runtime.")
    print("  Run the dependency-install cell, then re-run this GPU probe.")

PyTorch version: 2.9.0+cpu
CUDA available (safe probe): False


## Cell 3: Clone Repository from GitHub

In [2]:
# Clone or update repository (safe for "Run All")
from pathlib import Path
import os
import subprocess

CONFIG_YAML = "config" + ".yaml"
MAIN_PY = "main" + ".py"

cwd = Path.cwd()
# If we're already inside the repo, use current directory.
if (cwd / CONFIG_YAML).exists() and (cwd / "market_data").exists():
    repo_root = cwd
else:
    # Otherwise, assume repo lives at ./ml_engine (Colab default)
    repo_root = cwd / "ml_engine"

if not repo_root.exists():
    subprocess.run(["git", "clone", "https://github.com/Raynergy-svg/ml_engine.git"], check=True)
    print("✓ Repository cloned")
else:
    if (repo_root / ".git").exists():
        print("✓ Repository exists, updating (fast-forward only)...")
        try:
            subprocess.run(["git", "fetch", "origin", "main"], cwd=str(repo_root), check=True)
            subprocess.run(["git", "checkout", "main"], cwd=str(repo_root), check=True)
            subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=str(repo_root), check=True)
            print("✓ Repository updated")
        except subprocess.CalledProcessError:
            print("⚠️ Could not auto-update repo (possibly local edits in the runtime).")
            print("Proceeding with the existing checkout.")
    else:
        raise RuntimeError(f"Found {repo_root} but it's not a git repo")

%cd {repo_root}
print(f"✓ Working directory: {os.getcwd()}")


✓ Repository cloned
/content/ml_engine
✓ Working directory: /content/ml_engine


## Setup (Optional): Colab Secrets + Google Drive Persistence

- For FX dry-runs, set Colab **Secrets** (key icon): `OANDA_API_TOKEN` (or `OANDA_API_KEY`) and `OANDA_ACCOUNT_ID`.
- If you want artifacts/logs to persist across sessions, mount Google Drive.


In [ ]:
# Optional: mount Google Drive so trained_data/ survives runtime resets.
# To avoid "Run All" getting stuck waiting for auth, this is opt-in.

MOUNT_DRIVE = False  # <- set True to mount /content/drive
try:
    from google.colab import drive  # type: ignore
    _IN_COLAB = True
except Exception:
    drive = None
    _IN_COLAB = False

if not _IN_COLAB:
    print("Not running in Colab (skipping Drive mount).")
elif not MOUNT_DRIVE:
    print("Drive not mounted. Set MOUNT_DRIVE=True to mount /content/drive.")
else:
    try:
        # timeout_ms prevents indefinite waiting in some environments
        drive.mount("/content/drive", timeout_ms=60_000)
        print("✓ Drive mounted at /content/drive")
        print("Tip: you can copy trained_data/ to Drive at the end if desired.")
    except Exception as e:
        print(f"⚠️ Drive mount failed or timed out: {e}")
        print("Tip: re-run this cell and follow the auth prompt in Colab.")

KeyboardInterrupt: 

## Cell 6: Install Dependencies

In [ ]:
import sys
import subprocess


def pip_install(packages: list[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])


# Keep Colab's preinstalled CUDA torch if present; only install if missing.
try:
    import torch  # type: ignore

    print(f"PyTorch already installed: {torch.__version__}")
except Exception:
    print("PyTorch not found; installing torch...")
    pip_install(["torch"])


# Minimal deps for this notebook + FX guardrails/tests (avoid full requirements.txt on Colab).
pip_install(
    [
        "numpy",
        "pandas",
        "scikit-learn",
        "pyyaml",
        "tqdm",
        "rich",
        "matplotlib",
        "seaborn",
        "pytest",
        "python-dotenv",
        "tzdata",
        "requests",
    ]
)

print("✓ Dependencies installed")


## Buddy: OANDA Practice (Demo) — Connect + Trade

Run the next cell to verify OANDA practice connectivity (account summary + quote).

Then, only if you explicitly want the interactive REPL, set `RUN_BUDDY = True` in the following cell and run it. (This prevents "Run All" from getting stuck.)


In [ ]:
import os

# Best-effort: load Colab Secrets into env (no-op outside Colab)
def _maybe_load_colab_secrets() -> None:
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("OANDA_API_TOKEN") or userdata.get("OANDA_API_KEY")
        acct = userdata.get("OANDA_ACCOUNT_ID")
        if token and not (os.getenv("OANDA_API_TOKEN") or os.getenv("OANDA_API_KEY")):
            os.environ["OANDA_API_TOKEN"] = token
        if acct and not os.getenv("OANDA_ACCOUNT_ID"):
            os.environ["OANDA_ACCOUNT_ID"] = acct
    except Exception:
        return

_maybe_load_colab_secrets()

api_token = os.getenv("OANDA_API_TOKEN") or os.getenv("OANDA_API_KEY")
account_id = os.getenv("OANDA_ACCOUNT_ID")
print(f"OANDA token set: {bool(api_token)}")
print(f"OANDA account_id set: {bool(account_id)}")

if not api_token or not account_id:
    print("\nMissing OANDA secrets.")
    print("In Colab: click the key icon → set:")
    print("  - OANDA_API_TOKEN (or OANDA_API_KEY)")
    print("  - OANDA_ACCOUNT_ID")
else:
    from oanda_practice import OandaPracticeClient
    client = OandaPracticeClient.from_env()
    summary = client.get_account_summary()
    acct = (summary or {}).get("account") or {}
    print("\nAccount summary:")
    print(f"  id: {acct.get('id')}")
    print(f"  balance: {acct.get('balance')}")
    print(f"  NAV: {acct.get('NAV')}")
    print(f"  marginAvailable: {acct.get('marginAvailable')}")
    q = client.get_price_quote(instrument="EUR_USD")
    print(f"\nEUR_USD quote: bid={q['bid']:.6f} ask={q['ask']:.6f}")


In [ ]:
# Buddy REPL (interactive)
RUN_BUDDY = False  # <- set True to start Buddy

if not RUN_BUDDY:
    print("Buddy not started. Set RUN_BUDDY=True to launch the interactive REPL.")
elif (os.getenv("OANDA_API_TOKEN") or os.getenv("OANDA_API_KEY")) and os.getenv("OANDA_ACCOUNT_ID"):
    from main import buddy as _buddy
    _buddy(
        CONFIG_YAML,
        instrument="EUR_USD",
        granularity="M5",
        candles=300,
        execute=False,  # start in dry-run; type 'execute on' inside Buddy when ready
        verbose=True,
    )
else:
    print("Skipping Buddy start: missing OANDA secrets.")


## Setup: Validate Config + FX Guardrails

Runs quick checks so you catch config/FX policy issues before long training.


In [ ]:
from utils import load_config
from config_validator import validate_config

try:
    from fx_guardrails import FxPolicy
except Exception as e:
    FxPolicy = None
    print(f"⚠️ Could not import fx_guardrails: {e}")

config = load_config(CONFIG_YAML)

ok = validate_config(config, raise_on_error=False)
print(f"Config valid: {ok}")

fx_cfg = config.get("fx")
if not fx_cfg:
    print(f"⚠️ {CONFIG_YAML} has no 'fx:' section (FX Tier-1 guardrails need it)")
elif FxPolicy is not None:
    policy = FxPolicy.from_dict(fx_cfg)
    print("FX policy loaded:")
    print(f"  timezone: {policy.session.timezone}")
    print(f"  allowlist: {sorted(policy.allowlist_instruments)}")
    print(f"  trade window: {policy.session.trade_start}-{policy.session.trade_end}")
    print(f"  force-flat: {policy.session.force_flat_cutoff}")
    print(f"  max positions: {policy.limits.max_open_positions}")
    print(f"  max entries/day: {policy.limits.max_entries_per_day}")
    print(f"  practice_only: {policy.execution.practice_only}")
    print(f"  require_confirmation: {policy.execution.require_confirmation}")


## Setup: CLI Wiring Sanity Check

Confirms the notebook matches the current repo CLI entrypoints.


In [ ]:
import sys
import subprocess

MAIN_PY = "main.py"


def run(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)


run([sys.executable, MAIN_PY, "--help"])
run([sys.executable, MAIN_PY, "fx-paper", "--help"])


## FX: Tier‑1 Paper Dry‑Run (No Execution)

This runs the FX Tier‑1 guardrails workflow in **dry-run** mode. It will **not** place orders.


In [ ]:
import os
import sys
import subprocess


def _maybe_load_colab_secrets() -> None:
    try:
        from google.colab import userdata  # type: ignore

        token = userdata.get("OANDA_API_TOKEN") or userdata.get("OANDA_API_KEY")
        acct = userdata.get("OANDA_ACCOUNT_ID")
        if token and not (os.getenv("OANDA_API_TOKEN") or os.getenv("OANDA_API_KEY")):
            os.environ["OANDA_API_TOKEN"] = token
        if acct and not os.getenv("OANDA_ACCOUNT_ID"):
            os.environ["OANDA_ACCOUNT_ID"] = acct
    except Exception:
        return


_maybe_load_colab_secrets()

api_token = os.getenv("OANDA_API_TOKEN") or os.getenv("OANDA_API_KEY")
account_id = os.getenv("OANDA_ACCOUNT_ID")

if not api_token or not account_id:
    print("Skipping FX dry-run: missing OANDA secrets.")
    print("Set these in Colab -> Secrets (key icon):")
    print("  - OANDA_API_TOKEN (or OANDA_API_KEY)")
    print("  - OANDA_ACCOUNT_ID")
else:
    cmd = [
        sys.executable,
        MAIN_PY,
        "fx-paper",
        "--config",
        CONFIG_YAML,
        "--instrument",
        "EUR_USD",
        "--granularity",
        "M5",
        "--candles",
        "300",
    ]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)


## Quick Tests (FX Guardrails)

Runs a fast subset of unit tests (no training).


In [ ]:
import sys
import subprocess

cmd = [
    sys.executable,
    "-m",
    "pytest",
    "-q",
    "tests/test_fx_guardrails.py",
    "tests/test_fx_paper.py",
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=False)


## Cell 8: Update Config for GPU Training

In [ ]:
import yaml
import torch

# Load config
with open(CONFIG_YAML, "r") as f:
    config = yaml.safe_load(f)

# Update for GPU and faster training
cuda_ok = bool(globals().get("CUDA_AVAILABLE", False))
config["device"] = "cuda" if cuda_ok else "cpu"
config["batch_size"] = 128  # Larger batch size for GPU
config["epochs"] = 100  # Adjust as needed
config["auto_resume"] = True  # Enable auto-resume
config["mixed_precision"] = True  # Enable mixed precision for faster training
config["early_stopping_patience"] = 20

# Save updated config
with open(CONFIG_YAML, "w") as f:
    yaml.dump(config, f)

print("Config updated:")
print(f"  Device: {config['device']}")
print(f"  Batch size: {config['batch_size']}")
print(f"  Epochs: {config['epochs']}")


## Cell 10: Train the Model (With Technical Indicators)

This cell will:
- Load all CSV files from market_data/
- Enable feature engineering (SMA, EMA, RSI, MACD, Bollinger Bands, etc.)
- Add 50+ technical indicators for better predictions
- Train the model with enhanced features

In [ ]:
from ml_engine_enhanced import EnhancedMLEngine
from neural_network_integrator_enhanced import NeuralNetworkIntegrator
from mr_engine import MREngine
from reasoning_enhanced import ReasoningEngine
from data_loader import MarketDataLoader
from utils import load_config
import pandas as pd
import glob
import numpy as np
import torch

# Load config
config = load_config(CONFIG_YAML)

# Load market data from all CSV files (exclude predictions files)
print("Loading market data...")
csv_files = glob.glob('market_data/*.csv')
csv_files = [f for f in csv_files if 'predictions' not in f]
print(f"Found {len(csv_files)} CSV files")

if not csv_files:
    raise ValueError("No CSV files found in market_data/ folder.")

# Load and combine all CSV files with proper datetime handling
dfs = []
for file in csv_files:
    print(f"Loading {file}...")
    df_part = pd.read_csv(file)

    # Standardize column names to lowercase
    df_part.columns = df_part.columns.str.lower()

    # Convert date/datetime column to datetime and set as index (UTC -> tz-naive)
    if 'date' in df_part.columns:
        df_part['date'] = pd.to_datetime(df_part['date'], utc=True)
        df_part.set_index('date', inplace=True)
        # Convert to timezone naive for compatibility
        df_part.index = df_part.index.tz_convert(None)
    elif 'datetime' in df_part.columns:
        df_part['datetime'] = pd.to_datetime(df_part['datetime'], utc=True)
        df_part.set_index('datetime', inplace=True)
        df_part.index = df_part.index.tz_convert(None)

    # Ensure required columns exist
    required_cols = ['open', 'high', 'low', 'close', 'volume']
    missing_cols = [col for col in required_cols if col not in df_part.columns]
    if missing_cols:
        print(f"Warning: {file} missing columns {missing_cols}, skipping")
        continue

    dfs.append(df_part)

if not dfs:
    raise ValueError("No valid data frames after processing CSV files")

# Combine all dataframes
df = pd.concat(dfs, axis=0).sort_index()
print(f"Combined data shape: {df.shape}")

# Initialize data loader and preprocess
data_loader = MarketDataLoader(config)

# Preprocess data (add features and indicators)
print("Preprocessing data with technical indicators...")
preprocess_out = data_loader.preprocess(
    df,
    add_features=True,
    scaler_type="standard",
    sequence_length=config.get('sequence_length', 60),
    test_size=0.2,
    validation_size=0.1,
    use_cache=True
)

# Defensive unpacking (older/newer preprocess signatures)
if not (isinstance(preprocess_out, tuple) and len(preprocess_out) == 6):
    raise ValueError(f"Unexpected preprocess return: {type(preprocess_out)}")

X_train, y_train, X_val, y_val, X_test, y_test = preprocess_out
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape}, y={y_val.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

# Update model config with correct input size
config.setdefault('model', {})
config['model']['input_size'] = X_train.shape[2]

print("\n" + "="*60)
print("TRAINING")
print("="*60)

# Initialize and train the ML engine
ml_engine = EnhancedMLEngine(config)
result_initial = ml_engine.train(
    X_train, y_train,
    X_val, y_val,
    epochs=config['epochs']
 )
result = result_initial

print("\n" + "="*60)
print("INTEGRATION (ML + MT + MR)")
print("="*60)

# Build the 3-engine integrator used by the project.
# NOTE: We *train* the ML engine here; MT/MR are lightweight torch modules used for integrated predictions.
integrator_config = {
    'device': config.get('device', 'cuda' if torch.cuda.is_available() else 'cpu'),
    'use_attention': False,
    'use_dynamic_weights': True,
}

integrator = NeuralNetworkIntegrator(integrator_config)
mr_engine = MREngine(config.get('mr_engine', {}))
reasoning_engine = ReasoningEngine(config.get('reasoning', {}))

# For paper/demo usage: attach reasoning + MT/MR modules (if available)
integrator.set_engines(ml_engine.model, mr_engine.model, reasoning_engine)

# Quick smoke prediction on validation slice
try:
    _smoke = integrator.predict({
        'ml_features': torch.tensor(X_val[:1], dtype=torch.float32),
        'mt_features': torch.zeros((1, X_val.shape[1], 11), dtype=torch.float32),
        'mr_features': torch.zeros((1, X_val.shape[1], 11), dtype=torch.float32),
    })
    print("\nIntegrated smoke:")
    print(f"  prediction: {_smoke.get('prediction')}")
    print(f"  uncertainty: {_smoke.get('uncertainty')}")
    print(f"  weights (ml, mt, mr): {_smoke.get('weights')}")
    print(f"  integrated_pred shape: {np.asarray(_smoke.get('prediction')).shape}")
except Exception as e:
    print(f"Integrated smoke failed: {e}")

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Resumed: {result.get('resumed', False)}")
print(f"Total epochs: {result.get('total_epochs', 'N/A')}")
print(f"Best validation loss: {result['best_val_loss']:.6f}")
print(f"Final train loss: {result['train_losses'][-1]:.6f}")
print(f"Final val loss: {result['val_losses'][-1]:.6f}")

## Cell 11: Continue Training (Optional)

Runs extra training sessions to try to improve the current best checkpoint. (This runs before the download cell when you use "Run All".)

In [ ]:
import time

# Extra training loop (optional)
num_training_sessions = 5
epochs_per_session = 100

print("="*60)
print("CONTINUATION TRAINING")
print("="*60)
print(f"Sessions: {num_training_sessions}")
print(f"Epochs/session: {epochs_per_session}")
print(f"Total extra epochs: {num_training_sessions * epochs_per_session}")

result2 = None

for session in range(1, num_training_sessions + 1):
    print(f"\n{'='*60}")
    print(f"SESSION {session}/{num_training_sessions}")
    print(f"{'='*60}")

    start_time = time.time()
    result2 = ml_engine.train(
        X_train, y_train,
        X_val, y_val,
        epochs=epochs_per_session,
    )
    elapsed = time.time() - start_time

    # Keep `result` pointing at the most recent training run
    result = result2

    print(f"\nSession {session} done in {elapsed/60:.1f} min")
    print(f"  Resumed: {result.get('resumed', False)}")
    print(f"  Total epochs: {result.get('total_epochs', 'N/A')}")
    print(f"  Best val loss: {result['best_val_loss']:.6f}")
    print(f"  Final train loss: {result['train_losses'][-1]:.6f}")
    print(f"  Final val loss: {result['val_losses'][-1]:.6f}")

    if result.get('stopped_early', False):
        print("\n✓ Early stopping triggered")
        break

result_last = result
print("\n✓ Continuation training complete")
print("Best model path: trained_data/models/best_model.pth")
print("If you downloaded earlier, re-run the Download cell to get the latest checkpoint.")

## Cell 15: Evaluate Model (Most Recent Run)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _as_1d(a):
    a = np.asarray(a)
    return a.reshape(-1)

def _basic_metrics(y_true_1d: np.ndarray, y_pred_1d: np.ndarray):
    y_true_1d = _as_1d(y_true_1d)
    y_pred_1d = _as_1d(y_pred_1d)
    err = y_pred_1d - y_true_1d
    mse = float(np.mean(err ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(err)))
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((y_true_1d - float(np.mean(y_true_1d))) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float('nan')
    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2_score': r2}

# Evaluate ML engine (trained) on validation + test sets
ml_val_metrics = ml_engine.evaluate(X_val, y_val)
ml_test_metrics = ml_engine.evaluate(X_test, y_test)

print("ML Engine Validation Metrics:")
for key, value in ml_val_metrics.items():
    print(f"  {key}: {value:.6f}")

print("\nML Engine Test Metrics:")
for key, value in ml_test_metrics.items():
    print(f"  {key}: {value:.6f}")

# Evaluate integrated (ML+MT+MR) predictions
# Reuse helper defined in the training cell; provide a fallback for out-of-order runs
if "_as_11_features" not in globals():
    def _as_11_features(X: np.ndarray) -> np.ndarray:
        X = np.asarray(X)
        if X.ndim != 3:
            raise ValueError(f'Expected 3D tensor (batch, seq, feat), got shape={X.shape}')
        if X.shape[2] == 11:
            return X
        if X.shape[2] > 11:
            return X[:, :, :11]
        pad = np.zeros((X.shape[0], X.shape[1], 11 - X.shape[2]), dtype=X.dtype)
        return np.concatenate([X, pad], axis=2)

def _integrated_predict_batched(X: np.ndarray, batch_size: int, label: str):
    X = np.asarray(X)
    n = int(X.shape[0])
    preds = []
    weights_acc = []
    t0 = __import__('time').time()
    for start in range(0, n, batch_size):
        end = min(n, start + batch_size)
        xb = X[start:end]
        out = integrator.predict({
            'ml_features': xb,
            'mt_features': _as_11_features(xb),
            'mr_features': _as_11_features(xb),
        })
        pred = np.asarray(out.get('prediction'))
        preds.append(pred.reshape(pred.shape[0], -1))
        w = out.get('weights')
        if w is not None:
            w = np.asarray(w, dtype=float).reshape(-1)
            if w.size == 3:
                weights_acc.append(w)
        if start == 0 or end == n or (start // batch_size) % 10 == 0:
            elapsed = __import__('time').time() - t0
            print(f'  {label}: {end}/{n} ({100.0*end/n:.1f}%) in {elapsed:.1f}s')
    pred_all = np.concatenate(preds, axis=0).reshape(-1)
    w_mean = np.mean(np.stack(weights_acc, axis=0), axis=0) if weights_acc else None
    return pred_all, w_mean

print('\nIntegrated prediction (batched):')
batch_size = int(config.get('eval_batch_size', 2048))
pred_val_int_1d, w_val = _integrated_predict_batched(X_val, batch_size=batch_size, label='val')
pred_test_int_1d, w_test = _integrated_predict_batched(X_test, batch_size=batch_size, label='test')
y_val_1d = _as_1d(y_val)
y_test_1d = _as_1d(y_test)
int_val_metrics = _basic_metrics(y_val_1d, pred_val_int_1d)
int_test_metrics = _basic_metrics(y_test_1d, pred_test_int_1d)

print("\nIntegrated (ML+MT+MR) Validation Metrics:")
for key, value in int_val_metrics.items():
    print(f"  {key}: {value:.6f}")
print("\nIntegrated (ML+MT+MR) Test Metrics:")
for key, value in int_test_metrics.items():
    print(f"  {key}: {value:.6f}")
print(f"\nIntegrated engine weights (ml, mt, mr) avg: {w_val}")

loss_type = str(config.get("loss_type", "mse")).lower()
train_val_losses = result.get("val_losses", [])

# ML-only predictions (for training loss scale check)
pred_val_1d = _as_1d(ml_engine.predict(X_val))

print("\nPrediction stats (val):")
print(f"  y_val:   min={float(np.min(y_val_1d)):.6f} max={float(np.max(y_val_1d)):.6f} mean={float(np.mean(y_val_1d)):.6f} std={float(np.std(y_val_1d)):.6f}")
print(f"  pred:    min={float(np.min(pred_val_1d)):.6f} max={float(np.max(pred_val_1d)):.6f} mean={float(np.mean(pred_val_1d)):.6f} std={float(np.std(pred_val_1d)):.6f}")
print(f"  pred_int:min={float(np.min(pred_val_int_1d)):.6f} max={float(np.max(pred_val_int_1d)):.6f} mean={float(np.mean(pred_val_int_1d)):.6f} std={float(np.std(pred_val_int_1d)):.6f}")

# Recompute the configured loss on the full validation set (sanity check for scale)
err = pred_val_1d - y_val_1d
abs_err = np.abs(err)
recomputed_loss = None
if loss_type == "mse":
    recomputed_loss = float(np.mean(err ** 2))
elif loss_type == "mae":
    recomputed_loss = float(np.mean(abs_err))
elif loss_type in ("huber", "smooth_l1"):
    delta = float(config.get("huber_delta", 1.0)) if loss_type == "huber" else 1.0
    quad = 0.5 * (err ** 2)
    lin = delta * (abs_err - 0.5 * delta)
    recomputed_loss = float(np.mean(np.where(abs_err <= delta, quad, lin)))

if train_val_losses and recomputed_loss is not None:
    print(f"\nLoss scale check (loss_type={loss_type}):")
    print(f"  last val_loss (training): {train_val_losses[-1]:.6f}")
    print(f"  recomputed on y_val:       {recomputed_loss:.6f}")

# Visual sanity checks (integrated prediction)
# 1) Scatter uses a random sample across the *full* val set
seed = int(config.get("random_seed", 42))
rng = np.random.default_rng(seed)
m = min(2000, len(y_val_1d))
sample_idx = rng.choice(len(y_val_1d), size=m, replace=False)
y_s = y_val_1d[sample_idx]
p_s = pred_val_int_1d[sample_idx]

# 2) Series plot uses a window centered around the most "interesting" region (largest deviation from mean)
n = min(250, len(y_val_1d))
idx_peak = int(np.argmax(np.abs(y_val_1d - float(np.mean(y_val_1d)))))
start = max(0, min(len(y_val_1d) - n, idx_peak - n // 2))
end = start + n

lo = float(min(np.min(y_s), np.min(p_s)))
hi = float(max(np.max(y_s), np.max(p_s)))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_s, p_s, s=10, alpha=0.5)
plt.plot([lo, hi], [lo, hi], linestyle='--', linewidth=1)
plt.xlabel("Actual (y_val)")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted (val sample)")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(y_val_1d[start:end], label="Actual", linewidth=1)
plt.plot(pred_val_int_1d[start:end], label="Predicted (integrated)", linewidth=1)
plt.xlabel("Sample index")
plt.ylabel("Value")
plt.title(f"Val window [{start}:{end}] actual vs predicted")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


## Cell 17: Download Trained Model (Final Step)

This downloads a zip containing `trained_data/` and `config.yaml` after all training has completed.

In [ ]:
from pathlib import Path
import shutil
import time
import zipfile

# Colab download helper (safe fallback outside Colab)
try:
    from google.colab import files  # type: ignore

    _IN_COLAB = True
except Exception:
    files = None
    _IN_COLAB = False

# Bundle trained artifacts (runs last)
repo_root = Path.cwd()
trained_dir = repo_root / "trained_data"
if not trained_dir.exists():
    raise FileNotFoundError(f"Missing {trained_dir} - run training first")

best_model_path = trained_dir / "models" / "best_model.pth"
if best_model_path.exists():
    stat = best_model_path.stat()
    print("Best model file:")
    print(f"  path (runtime): {best_model_path}")
    print("  path (relative): trained_data/models/best_model.pth")
    print(f"  size: {stat.st_size / (1024**2):.2f} MB")
    print(f"  modified: {time.ctime(stat.st_mtime)}")
else:
    print(f"Warning: best model file not found at {best_model_path}")

# Optional: print which training result is being exported
try:
    print("Training summary (most recent run):")
    print(f"  total_epochs: {result.get('total_epochs', 'N/A')}")
    print(f"  best_val_loss: {result.get('best_val_loss', float('nan')):.6f}")
except Exception:
    pass

export_dir = repo_root / "export_trained_artifacts"
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

# Include trained_data + config.yaml
shutil.copytree(trained_dir, export_dir / "trained_data")
shutil.copy2(repo_root / CONFIG_YAML, export_dir / CONFIG_YAML)

zip_path = repo_root / f"trained_artifacts_{int(time.time())}.zip"

print(f"\nCreating zip at: {zip_path}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in export_dir.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(export_dir))

print(f"\nZip created (runtime): {zip_path}")
print(f"Zip size: {zip_path.stat().st_size / (1024**2):.2f} MB")
print("\nNote: this path is inside the notebook runtime (e.g. Colab /content).")
print("The download will appear in your computer's Downloads folder (browser default).")

with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    must_have = [
        CONFIG_YAML,
        "trained_data/models/best_model.pth",
    ]
    print("\nZip content check:")
    for p in must_have:
        print(f"  {'✓' if p in names else '✗'} {p}")
    print("\nFirst 25 entries in zip:")
    for n in names[:25]:
        print(f"  {n}")

if _IN_COLAB:
    files.download(str(zip_path))
    print(f"\nTriggered browser download: {zip_path.name}")
else:
    print(f"\nNot running in Colab; zip is on disk at: {zip_path}")
